In [18]:
import torch
import torch.nn as nn
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

In [19]:
class PolicyNet(nn.Module):
    def __init__(self, input_dim: int, hidden=(10,10), output_dim: int = 1,
                 floor_mode: str = "max", min_bid: float = 1e-6):
        super().__init__()
        layers = []
        d = int(input_dim)
        for h in hidden:
            layers += [nn.Linear(d, int(h)), nn.ReLU()]
            d = int(h)
        layers += [nn.Linear(d, int(output_dim))]
        self.net = nn.Sequential(*layers)

        # parámetros “no entrenables” (puedes cambiar por llamada)
        self.floor_mode = floor_mode
        self.min_bid = float(min_bid)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Devuelve BID (no raw):
          bid >= 0
          bid >= phi  (si x es phi escalar) o bid >= max_h phi_h (si x es phi vector 24)
        """
        raw = self.net(x).reshape(-1)          # (H,)
        inc = 0.02 * F.softplus(raw)                  # >= 0

        if x.ndim == 2 and x.shape[1] > 1:
            if self.floor_mode == "max":
                floor = x.max(dim=1).values   # (H,)
            elif self.floor_mode == "mean":
                floor = x.mean(dim=1)         # (H,)
            else:
                raise ValueError("floor_mode must be 'max' or 'mean'")
        else:
            floor = x.reshape(-1)             # (H,)

        floor = torch.clamp_min(floor, 0.0)   # >= 0 y >= floor original
        bid = floor + inc + self.min_bid      # >= 0 y >= floor

        return bid                             # (H,)


In [20]:
path_uniform = "symmetric/36redesmedian/nets_best_by_gid_uniform.pt"
saved_nets_uniform = torch.load(path_uniform, map_location="cpu")

gid = 1
net_uniform = PolicyNet(
    input_dim=24,
    hidden=(10,10),
    output_dim=1,
    floor_mode="mean",   # <- igual que en entrenamiento
    min_bid=1e-6         # <- igual que en entrenamiento
)
net_uniform.load_state_dict(saved_nets_uniform[gid], strict=True)
net_uniform.eval() # importante para modo evaluación



path_discriminatory = "symmetric/36redesmedian/nets_best_by_gid_pay_as_bid.pt"
saved_nets_discriminatory = torch.load(path_discriminatory, map_location="cpu")

gid = 1
net_discriminatory = PolicyNet(
    input_dim=24,
    hidden=(10,10),
    output_dim=1,
    floor_mode="mean",   # <- igual que en entrenamiento
    min_bid=1e-6         # <- igual que en entrenamiento
)
net_discriminatory.load_state_dict(saved_nets_discriminatory[gid], strict=True)
net_discriminatory.eval() # importante para modo evaluación

C:\Users\HP\AppData\Local\Temp\ipykernel_6724\1089277082.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_nets_uniform = torch.load(path_uniform, map_location="cpu"

PolicyNet(
  (net): Sequential(
    (0): Linear(in_features=24, out_features=10, bias=True)
    (1): ReLU()
    (2): Linear(in_features=10, out_features=10, bias=True)
    (3): ReLU()
    (4): Linear(in_features=10, out_features=1, bias=True)
  )
)

In [21]:
CSV_PATH = "symmetric/grupos.csv"   # <-- adjust
input = pd.read_csv(CSV_PATH)


In [22]:
input

,id_planta,Fecha,phi,Grupo,precio_d
0,1,2025-04-01,0.083924,1,0.10137
1,2,2025-04-01,0.262222,5,0.28295
2,3,2025-04-01,0.275288,5,0.32637
3,4,2025-04-01,0.260380,5,0.31037
4,5,2025-04-01,0.968472,4,1.02813
...,...,...,...,...,...
4207,32,2025-09-09,0.779362,4,0.90894
4208,33,2025-09-09,1.058581,4,1.11735
4209,34,2025-09-09,0.945489,4,1.01274
4210,35,2025-09-09,0.969786,4,1.18348


In [23]:
import torch

# Suponiendo que tu DataFrame se llama input
def phi_a_tensor_24(phi_val):
    """Recibe un valor escalar y devuelve un tensor (1,24) duplicado"""
    arr = [phi_val] * 24  # lista de 24 valores iguales
    return torch.tensor([arr], dtype=torch.float)  # shape (1,24)
# Lista para guardar resultados

predicciones_uniform = []
predicciones_discriminatory = []
for i, row in input.iterrows():
    x = phi_a_tensor_24(row['phi'])
    y_pred_uniform = net_uniform(x)
    y_pred_discrimininatoty = net_discriminatory(x)
    predicciones_uniform.append(y_pred_uniform.item())  # convertir de tensor a float
    predicciones_discriminatory.append(y_pred_discrimininatoty.item()) 

In [24]:
predicciones_uniform

[0.09512508660554886,
 0.2734285891056061,
 0.286495566368103,
 0.2715866267681122,
 0.9796948432922363,
 0.3013453483581543,
 0.10704254359006882,
 0.0968257412314415,
 0.22853077948093414,
 0.24241261184215546,
 0.26769283413887024,
 0.27542468905448914,
 0.2343859225511551,
 0.33771657943725586,
 0.3768555521965027,
 0.8726194500923157,
 0.8790159821510315,
 0.23777641355991364,
 0.5152719020843506,
 0.5037209391593933,
 1.3679075241088867,
 1.1991376876831055,
 0.23777641355991364,
 0.04785288870334625,
 2.3026158809661865,
 0.35653454065322876,
 1.7117218971252441,
 1.6157928705215454,
 0.8421642780303955,
 0.7259947657585144,
 0.7637630701065063,
 0.9197182059288025,
 1.4242277145385742,
 1.4305260181427002,
 1.1289440393447876,
 1.3680684566497803,
 0.08295868337154388,
 0.9578558206558228,
 0.23951295018196106,
 0.23478786647319794,
 0.9842351675033569,
 0.29632818698883057,
 0.10244853794574738,
 0.09546412527561188,
 0.20297570526599884,
 0.2058132141828537,
 0.24792341887950

In [25]:
predicciones_discriminatory

[0.09697728604078293,
 0.2752673327922821,
 0.28833338618278503,
 0.27342551946640015,
 0.9814954400062561,
 0.30318230390548706,
 0.10889395326375961,
 0.09867782890796661,
 0.23037338256835938,
 0.24425402283668518,
 0.26953205466270447,
 0.27726325392723083,
 0.23622801899909973,
 0.33955079317092896,
 0.3786875903606415,
 0.8744255900382996,
 0.8808217644691467,
 0.2396182268857956,
 0.5170966982841492,
 0.5055463314056396,
 1.3697011470794678,
 1.2009345293045044,
 0.2396182268857956,
 0.049708135426044464,
 2.3043510913848877,
 0.3583676218986511,
 1.7135028839111328,
 1.6175789833068848,
 0.8439719676971436,
 0.7278085350990295,
 0.7655748724937439,
 0.9215219020843506,
 1.4260202646255493,
 1.4323184490203857,
 1.1307423114776611,
 1.3698620796203613,
 0.08481168746948242,
 0.9596575498580933,
 0.24135461449623108,
 0.23662994801998138,
 0.9860357046127319,
 0.2981654405593872,
 0.10430025309324265,
 0.09731630980968475,
 0.20482052862644196,
 0.20765778422355652,
 0.2497643530

In [26]:
input['est_uni']=predicciones_uniform
input['est_dis']=predicciones_discriminatory

In [27]:
input

,id_planta,Fecha,phi,Grupo,precio_d,est_uni,est_dis
0,1,2025-04-01,0.083924,1,0.10137,0.095125,0.096977
1,2,2025-04-01,0.262222,5,0.28295,0.273429,0.275267
2,3,2025-04-01,0.275288,5,0.32637,0.286496,0.288333
3,4,2025-04-01,0.260380,5,0.31037,0.271587,0.273426
4,5,2025-04-01,0.968472,4,1.02813,0.979695,0.981495
...,...,...,...,...,...,...,...
4207,32,2025-09-09,0.779362,4,0.90894,0.790581,0.792391
4208,33,2025-09-09,1.058581,4,1.11735,1.069806,1.071605
4209,34,2025-09-09,0.945489,4,1.01274,0.956712,0.958513
4210,35,2025-09-09,0.969786,4,1.18348,0.981009,0.982809


## Equilibrios

In [11]:
import torch
import torch.nn.functional as F

x = phi_a_tensor_24(1.255597)
with torch.no_grad():
    raw = net_uniform.net(x).reshape(-1)
    inc = F.softplus(raw)
    bid = net_uniform(x).reshape(-1)

print("raw:", raw.item())
print("softplus(raw):", inc.item())
print("bid:", bid.item(), "phi:", x.mean().item(), "bid-phi:", (bid - x.mean()).item())

raw: 0.43513038754463196
softplus(raw): 0.9341952800750732
bid: 1.2742817401885986 phi: 1.2555968761444092 bid-phi: 0.018684864044189453


In [12]:
import torch
import torch.nn.functional as F

x = phi_a_tensor_24(1.255597)
with torch.no_grad():
    raw = net_discriminatory.net(x).reshape(-1)
    inc = F.softplus(raw)
    bid = net_discriminatory(x).reshape(-1)

print("raw:", raw.item())
print("softplus(raw):", inc.item())
print("bid:", bid.item(), "phi:", x.mean().item(), "bid-phi:", (bid - x.mean()).item())

raw: 0.3959486484527588
softplus(raw): 0.910591721534729
bid: 1.2738096714019775 phi: 1.2555968761444092 bid-phi: 0.01821279525756836


## notas:
en maximo da bien, uniform es mas pequeño que discirminatory bien

In [13]:
path_uniform = "symmetric/5redespromedio/nets_best_by_gid_uniform.pt"
saved_nets_uniform = torch.load(path_uniform, map_location="cpu")

gid = 1
net_uniform = PolicyNet(
    input_dim=24,
    hidden=(10,10),
    output_dim=1,
    floor_mode="mean",   # <- igual que en entrenamiento
    min_bid=1e-6         # <- igual que en entrenamiento
)
net_uniform.load_state_dict(saved_nets_uniform[gid], strict=True)
net_uniform.eval() # importante para modo evaluación



path_discriminatory = "symmetric/5redespromedio/nets_best_by_gid_pay_as_bid.pt"
saved_nets_discriminatory = torch.load(path_discriminatory, map_location="cpu")

gid = 1
net_discriminatory = PolicyNet(
    input_dim=24,
    hidden=(10,10),
    output_dim=1,
    floor_mode="mean",   # <- igual que en entrenamiento
    min_bid=1e-6         # <- igual que en entrenamiento
)
net_discriminatory.load_state_dict(saved_nets_discriminatory[gid], strict=True)
net_discriminatory.eval() # importante para modo evaluación


import torch
import torch.nn.functional as F

x = phi_a_tensor_24(1.255597)
with torch.no_grad():
    raw = net_uniform.net(x).reshape(-1)
    inc = F.softplus(raw)
    bid = net_uniform(x).reshape(-1)

print("raw:", raw.item())
print("softplus(raw):", inc.item())
print("bid:", bid.item(), "phi:", x.mean().item(), "bid-phi:", (bid - x.mean()).item())

x = phi_a_tensor_24(1.255597)
with torch.no_grad():
    raw = net_discriminatory.net(x).reshape(-1)
    inc = F.softplus(raw)
    bid = net_discriminatory(x).reshape(-1)

print("raw:", raw.item())
print("softplus(raw):", inc.item())
print("bid:", bid.item(), "phi:", x.mean().item(), "bid-phi:", (bid - x.mean()).item())

raw: 0.2204907089471817
softplus(raw): 0.8094573020935059
bid: 1.2717869281768799 phi: 1.2555968761444092 bid-phi: 0.016190052032470703
raw: 0.3046151101589203
softplus(raw): 0.8570089340209961
bid: 1.272737979888916 phi: 1.2555968761444092 bid-phi: 0.017141103744506836


C:\Users\HP\AppData\Local\Temp\ipykernel_6724\4167848203.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_nets_uniform = torch.load(path_uniform, map_location="cpu"